# DESC Supernovae counts (SNNSNMetric) versus the E(B-V) footprint threshold - figures replotted from cache

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-26
- Last update : 2026-09-28 : Normalize more correctly medians bu baseline reference
- Kernel: conda_py313_opsim53
- Context: SCOC footprint-shrinking study + DESC static-probes metrics.
- This notebook: **08** of the series - **regenerates every figure and table of `07_SNCounts_DESC_TaskForce_demo.ipynb`
  by reading back the `.npz` maps it cached**, without rerunning the (slow) `SNNSNMetric` MAF computation.
  `SNNSNMetric` simulates light curves pixel by pixel and season by season, so re-executing notebook 07 end to
  end just to tweak a plot or a caption is expensive; this notebook lets that iteration happen at (near) zero
  cost. All of the explanatory text of notebook 07 is kept, section by section, so this notebook can be read on
  its own.
- **Source of the data**: `data_07_SNCOUNTS/` (must already contain, for every run, the `n_sn` and `zlim`
  `.npz` files produced by a prior execution of notebook 07 with the configuration set in Section 3 below -
  see the practical note). This notebook never calls `MetricBundleGroup.run_all()`.
- Companion notebooks: `01_compareExgalM5withCuts.ipynb`, `03_WeakLensing.ipynb`,
  `06_FOM3x2pts_DESC_TaskForce_demo.ipynb` (same 7 runs, other DESC metrics), and of course
  `07_SNCounts_DESC_TaskForce_demo.ipynb` itself (the notebook that produced the cache read here, and the one
  to rerun if the configuration of Section 3 changes or a run is missing).
- OpSim simulations analyzed (`/Users/dagoret/DATA/OpSim/`) - listed here for reference only, this notebook does
  not open any of these database files:
  - `shrink_fp_dust_0.050_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.080_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.120_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.150_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.199_v5.3.6_10yrs.db`
  - `baseline_v5.3.6_10yrs.db` (E(B-V) threshold 0.200)
  - `shrink_fp_dust_0.250_v5.3.6_10yrs.db`


## Notebook overview

**What `SNNSNMetric` computes.** For every Healpix pixel and observing season it simulates fast SALT2-like Type Ia light curves from the actual cadence and per-visit depths, selects the ones that are *well measured* (at least `n_bef` points before peak and `n_aft` after, above a minimum SNR), finds the redshift completeness limit `zlim` (the redshift below which a fraction `zlim_coeff` of simulated SNeIa are well measured), and counts the expected number of well-measured SNeIa `n_sn` up to `zlim` from a volumetric rate model. MAF's reduce-function mechanism splits the (`n_sn`, `zlim`) output into two Healpix maps, `SNNSNMetric_reducen_sn` and `SNNSNMetric_reducezlim` - see `../06_MAF_DESC_TaskF/03_SN_DESC_TaskForce_demo.ipynb` (Section on "what counting supernovae means") for the full description and references.

**Choices made when the cache was produced (Section 3 below must match `07_SNCounts_DESC_TaskForce_demo.ipynb` exactly, since the cache file names encode the configuration).**
- `nside = 32` (a compromise between the official `science_radar_batch` value, `nside = 16`, and the `nside = 64` used elsewhere in this series; `SNNSNMetric` is far more expensive per pixel than the depth/visit-counting metrics of notebooks 01, 03 and 06, hence the coarser grid).
- non-DD visits only (`scheduler_note not like 'DD%'`, the convention of this `10_DESCMAFDEPTHANDGALCOUNT` series - the official `06_MAF_DESC_TaskF` demo uses the wider `'%DD%'` pattern, see the caveats), truncated to the **first 10 years** of every run (`night <= 10*365.25 + 0.5`), needed for the same reason as notebooks 01/03/06: the baseline file is `baseline_v5.3.6_11yrs`.
- `n_bef = 3`, `n_aft = 8`, `zmin = 0.2`, `zmax = 0.5`, `z_step = 0.03`, `daymax_step = 3.0`, `zlim_coeff = 0.95`, `gamma_name = "gamma_WFD.hdf5"`, `coadd_night = True`, `add_dust = False` - all copied verbatim from `../06_MAF_DESC_TaskF/03_SN_DESC_TaskForce_demo.ipynb` (the official `science_radar_batch` "SNe Ia" configuration).
- The dust cut is `SNNSNMetric`'s own `hard_dust_cut` parameter (pixels with `E(B-V) > hard_dust_cut` are masked entirely). As in notebooks 01, 03 and 06, `EBV_CUT_MODE = 'run'` (default) sets `hard_dust_cut` to the E(B-V) threshold that defines the WFD footprint of each run; `EBV_CUT_MODE = 'fixed'` uses the same `FIXED_LIM_EBV` for every run. Note that the *official* default of `hard_dust_cut` is **0.25**, not the `0.2` used as `FIXED_LIM_EBV` in notebooks 01/03 - `FIXED_LIM_EBV` here defaults to `0.25` to match `03_SN_DESC_TaskForce_demo.ipynb` exactly when `EBV_CUT_MODE = 'fixed'`.

**How the figures are built** (identical to notebook 07, applied to both `n_sn` and `zlim`).
1. Healpix map of `n_sn` and of `zlim` for each of the 7 runs, on one common color scale.
2. Area-weighted histograms of the valid pixels.
3. Difference maps between **consecutive** thresholds. `n_sn` is an extensive per-pixel count, so a masked pixel counts as **0** in the differences (as for the WL metrics of notebook 03); `zlim` is an intensive redshift limit with no natural "zero", so its differences are restricted to pixels valid in **both** runs (as for the depth of notebook 01). The pixels gained or lost by the footprint are shown separately.
4. Total `n_sn` (the headline DESC SN count) and median/mean `zlim`, with the pixel-to-pixel dispersion, versus the E(B-V) threshold, and the normalized versions.

**Caching.** This notebook only *reads* the `.npz` caches of `data_07_SNCOUNTS/` (never writes a map, never runs MAF); it recomputes only the lightweight summary tables and figures from them, and writes those (CSV, PNG, PDF) to `data_08_SNCOUNTS_REPLOT/` and `figs_08_SNCOUNTS_REPLOT/`, so that rerunning this notebook never touches notebook 07's own outputs.

## Practical note before running

This notebook does **not** need `george`/`scikit-learn`-style heavy fitting nor a working OpSim database connection: it only loads the `.npz` files that `07_SNCounts_DESC_TaskForce_demo.ipynb` already saved in `data_07_SNCOUNTS/`. If Section 6 below raises a `FileNotFoundError` for a given run, that run (under the exact configuration of Section 3: `nside`, `n_bef`, `n_aft`, `zmin`/`zmax`, and `hard_dust_cut`) has not been computed yet - go run notebook 07 for it first (it will only recompute the missing runs, not the ones already cached).

## 1. Imports

In [ ]:
import os
import inspect
from os.path import join, isfile

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import rubin_sim
from rubin_sim.maf.metrics.sn_n_sn_metric import SNNSNMetric

print("rubin_sim version:", rubin_sim.__version__)

## 2. Directories

In [ ]:
# Where notebook 07 cached its n_sn/zlim maps (read-only for this notebook)
SRC_DATA_DIR = "data_07_SNCOUNTS"

# This notebook's own outputs (regenerated summary tables and figures)
NB_TAG = "SNCOUNTS_REPLOT"
data_dir = f"data_08_{NB_TAG}"
figs_dir = f"figs_08_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("Reading cached maps from :", os.path.abspath(SRC_DATA_DIR))
print("Regenerated tables (data):", os.path.abspath(data_dir))
print("Regenerated figures      :", os.path.abspath(figs_dir))

## 3. Configuration

In [ ]:
# (run_name, E(B-V) threshold that defines the footprint of the run), sorted by increasing threshold
# - identical list to notebooks 01, 03, 06 and 07 of this series
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_10yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
DUST_THRESH = dict(RUNS_INFO)
REF_RUN = "baseline_v5.3.6_10yrs"

# Consecutive-threshold pairs (later run minus earlier run)
PAIRS = [(RUN_NAMES[i + 1], RUN_NAMES[i]) for i in range(len(RUN_NAMES) - 1)]

# Must match the values used when notebook 07 produced the cache (they select the cache file name)
SNE_NSIDE = 32
N_BEF, N_AFT = 3, 8
ZMIN, ZMAX = 0.2, 0.5

EBV_CUT_MODE = "run"  # 'run' or 'fixed' - see notebook 07, Section 3
FIXED_LIM_EBV = 0.25

pix_area = hp.nside2pixarea(SNE_NSIDE, degrees=True)
print(
    f"nside={SNE_NSIDE} -> npix={hp.nside2npix(SNE_NSIDE)}, pixel area = {pix_area:.3f} deg^2 "
    f"(official batch: nside=16, pixel area = {hp.nside2pixarea(16, degrees=True):.2f} deg^2)"
)


def metric_ebv_cut(run_name):
    if EBV_CUT_MODE == "run":
        return DUST_THRESH[run_name]
    if EBV_CUT_MODE == "fixed":
        return FIXED_LIM_EBV
    raise ValueError(f"EBV_CUT_MODE must be 'run' or 'fixed', got {EBV_CUT_MODE!r}")


def config_tag(run_name):
    ebv = metric_ebv_cut(run_name)
    return f"nside{SNE_NSIDE}_nbef{N_BEF}_naft{N_AFT}_z{ZMIN:.2f}-{ZMAX:.2f}_hdc{ebv:.3f}"


print(f"hard_dust_cut expected in the cache (mode '{EBV_CUT_MODE}'):")
for run_name, _ in RUNS_INFO:
    print(
        f"  {run_name:40s} hard_dust_cut = {metric_ebv_cut(run_name):.3f}   cache tag: {config_tag(run_name)}"
    )

## 4. The metric class used

Docstring and the reduce functions that split the (`n_sn`, `zlim`) output into two independent Healpix maps (unchanged from notebook 07 - kept here for reference, this notebook does not instantiate or run the metric).

In [ ]:
print(inspect.getdoc(SNNSNMetric))
print("-" * 80)
print("reduce_funcs:", list(SNNSNMetric().reduce_funcs.keys()))

## 5. Helper functions

Same conventions as notebooks 01, 03 and 07 (masked-array maps, `hp.UNSEEN` for masked pixels, PNG+PDF saving), applied here to two kinds of map (`NSN`, `ZLIM`) loaded from the cache of notebook 07 rather than freshly computed.

In [ ]:
def as_healpy(m):
    return np.ma.filled(m, hp.UNSEEN)


def run_label(run_name):
    label = f"E(B-V) < {DUST_THRESH[run_name]:.3f}"
    return label + " (baseline)" if run_name.startswith("baseline") else label


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


def load_map(path):
    with np.load(path) as f:
        return np.ma.masked_array(f["data"], mask=f["mask"])


def msum(m):
    return float(np.ma.getdata(m)[~np.ma.getmaskarray(m)].sum())


def diff_map(map_b, map_a, fill_masked=None):
    mask_b, mask_a = np.ma.getmaskarray(map_b), np.ma.getmaskarray(map_a)
    b = np.ma.getdata(map_b).astype(float)
    a = np.ma.getdata(map_a).astype(float)
    if fill_masked is None:
        return np.ma.masked_array(b - a, mask=mask_b | mask_a)
    b = np.where(mask_b, fill_masked, b)
    a = np.where(mask_a, fill_masked, a)
    return np.ma.masked_array(b - a, mask=mask_b & mask_a)


def pair_stats(map_b, map_a, dmap, with_totals=False):
    vb, va = ~np.ma.getmaskarray(map_b), ~np.ma.getmaskarray(map_a)
    common, gained, lost = vb & va, vb & ~va, va & ~vb
    d = dmap.compressed()
    row = {
        "area_a_deg2": va.sum() * pix_area,
        "area_b_deg2": vb.sum() * pix_area,
        "area_gained_deg2": gained.sum() * pix_area,
        "area_lost_deg2": lost.sum() * pix_area,
        "diff_mean": d.mean() if d.size else np.nan,
        "diff_median": np.median(d) if d.size else np.nan,
        "diff_std": d.std() if d.size else np.nan,
        "diff_min": d.min() if d.size else np.nan,
        "diff_max": d.max() if d.size else np.nan,
    }
    if with_totals:
        b = np.ma.getdata(map_b).astype(float)
        a = np.ma.getdata(map_a).astype(float)
        row.update(
            {
                "total_a": a[va].sum(),
                "total_b": b[vb].sum(),
                "total_diff": b[vb].sum() - a[va].sum(),
                "from_common_pixels": (b[common] - a[common]).sum(),
                "from_gained_pixels": b[gained].sum(),
                "from_lost_pixels": -a[lost].sum(),
            }
        )
    return row


def per_run_summary(maps, with_sum=False):
    rows = []
    for run_name, dust in RUNS_INFO:
        v = maps[run_name].compressed()
        row = {
            "run": run_name,
            "E(B-V) cut of the run": dust,
            "hard_dust_cut of the metric": metric_ebv_cut(run_name),
            "n_pixels": v.size,
            "area_deg2": v.size * pix_area,
            "mean": v.mean() if v.size else np.nan,
            "median": np.median(v) if v.size else np.nan,
            "std": v.std() if v.size else np.nan,
            "min": v.min() if v.size else np.nan,
            "max": v.max() if v.size else np.nan,
        }
        if with_sum:
            row["sum"] = v.sum()
        rows.append(row)
    return pd.DataFrame(rows).set_index("run")


def plot_all_maps(maps, tag, unit, title, cmap="viridis"):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    if pooled.size == 0:
        print(f"{title}: all pixels are masked in all runs, nothing to plot")
        return
    vmin, vmax = np.percentile(pooled, 1), np.percentile(pooled, 99)
    fig = plt.figure(figsize=(15, 12))
    for i, (run_name, dust) in enumerate(RUNS_INFO, start=1):
        hp.mollview(
            as_healpy(maps[run_name]),
            fig=fig.number,
            sub=(3, 3, i),
            min=vmin,
            max=vmax,
            cmap=cmap,
            title=f"{run_name}\n{run_label(run_name)}",
            unit=unit,
            cbar=True,
        )
    fig.suptitle(f"{title} (common color scale: 1st-99th percentile of all runs)", fontsize=16, y=1.02)
    save_fig(fig, f"{tag}_healpix_allruns")
    plt.show()


def plot_overlay_hist(maps, tag, xlabel, title, nbins=60):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    if pooled.size == 0:
        print(f"{title}: all pixels are masked in all runs, nothing to plot")
        return
    bins = np.linspace(np.percentile(pooled, 0.1), np.percentile(pooled, 99.9), nbins + 1)
    colors = plt.cm.viridis(np.linspace(0.0, 0.95, len(RUNS_INFO)))
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    for (run_name, dust), c in zip(RUNS_INFO, colors):
        v = maps[run_name].compressed()
        w = np.full(v.size, pix_area)
        axs[0].hist(v, bins=bins, weights=w, histtype="step", color=c, label=run_label(run_name))
        axs[1].hist(
            v, bins=bins, weights=w, histtype="step", color=c, cumulative=-1, label=run_label(run_name)
        )
    axs[0].set_ylabel("Area per bin [deg\u00b2]")
    axs[1].set_ylabel("Area with value \u2265 x [deg\u00b2]")
    for ax in axs:
        ax.set_xlabel(xlabel)
        ax.grid(alpha=0.3)
    axs[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    save_fig(fig, f"{tag}_histograms_allruns")
    plt.show()


def plot_diff_mosaic(diff_maps, tag, unit, label):
    lims = [np.percentile(np.abs(d.compressed()), 99) for d in diff_maps.values() if d.count()]
    vlim = max(max(lims), 1e-6) if lims else 1.0
    fig = plt.figure(figsize=(16, 10))
    for i, ((run_b, run_a), d) in enumerate(diff_maps.items(), start=1):
        hp.mollview(
            as_healpy(d),
            fig=fig.number,
            sub=(2, 3, i),
            min=-vlim,
            max=vlim,
            cmap="RdBu_r",
            title=f"{run_b}\nminus {run_a}",
            unit=unit,
            cbar=True,
        )
    fig.suptitle(
        f"{label}: differences between consecutive dust thresholds (common scale \u00b1{vlim:.3g})",
        fontsize=16,
        y=1.02,
    )
    save_fig(fig, f"{tag}_diff_allpairs_combined")
    plt.show()


def plot_footprint_change(maps, tag, label):
    cmap = ListedColormap(["tab:red", "lightgreen", "tab:blue"])
    fig = plt.figure(figsize=(16, 10))
    for i, (run_b, run_a) in enumerate(PAIRS, start=1):
        vb, va = ~np.ma.getmaskarray(maps[run_b]), ~np.ma.getmaskarray(maps[run_a])
        status = np.zeros(vb.size)
        status[vb & ~va] = 1.0
        status[va & ~vb] = -1.0
        status = np.ma.masked_array(status, mask=~(va | vb))
        hp.mollview(
            as_healpy(status),
            fig=fig.number,
            sub=(2, 3, i),
            min=-1,
            max=1,
            cmap=cmap,
            cbar=False,
            title=f"{run_b}\nminus {run_a}",
        )
    fig.suptitle(
        f"{label}: footprint change between consecutive dust thresholds "
        "(blue = pixel gained, red = pixel lost, green = valid in both, dark grey = valid in neither)",
        fontsize=16,
        y=1.02,
    )
    save_fig(fig, f"{tag}_footprint_change_combined")
    plt.show()


def plot_diff_histograms(diff_maps, tag, xlabel, label):
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for ax, ((run_b, run_a), d) in zip(axes.flat, diff_maps.items()):
        ax.hist(d.compressed(), bins=80, color="steelblue")
        ax.axvline(0, color="k", linewidth=0.8)
        ax.set_title(f"{run_b}\n- {run_a}", fontsize=9)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("N pixels")
    fig.suptitle(f"{label}: histograms of the consecutive differences")
    fig.tight_layout()
    save_fig(fig, f"{tag}_diff_histograms")
    plt.show()


def analyse_pairs(maps, tag, label, unit, xlabel, fill_masked=None, with_totals=False):
    diff_maps, rows = {}, []
    for run_b, run_a in PAIRS:
        d = diff_map(maps[run_b], maps[run_a], fill_masked=fill_masked)
        diff_maps[(run_b, run_a)] = d
        row = {"pair (E(B-V) thresholds)": f"{DUST_THRESH[run_b]:.3f} - {DUST_THRESH[run_a]:.3f}"}
        row.update(pair_stats(maps[run_b], maps[run_a], d, with_totals=with_totals))
        rows.append(row)
    plot_diff_mosaic(diff_maps, tag, unit, label)
    plot_footprint_change(maps, tag, label)
    plot_diff_histograms(diff_maps, tag, xlabel, label)
    stats = pd.DataFrame(rows).set_index("pair (E(B-V) thresholds)")
    stats.to_csv(join(data_dir, f"{tag}_diff_summary.csv"))
    print("Saved:", join(data_dir, f"{tag}_diff_summary.csv"))
    return diff_maps, stats

## 6. Load the cached `n_sn` and `zlim` maps of each run (no MAF rerun)

Each map was cached by notebook 07 as its own `.npz` in `data_07_SNCOUNTS/`; a missing file means that run has not been computed yet under the configuration of Section 3 above - go run notebook 07 for it (it only recomputes the missing runs).

In [ ]:
def cache_path(kind, run_name):
    tag = run_name.replace(".", "_")
    return join(SRC_DATA_DIR, f"SNNSN_{kind}_{tag}_{config_tag(run_name)}.npz")


def load_run(run_name):
    paths = {kind: cache_path(kind, run_name) for kind in ("nsn", "zlim")}
    missing = [p for p in paths.values() if not isfile(p)]
    if missing:
        raise FileNotFoundError(
            f"[{run_name}] missing cached map(s): {missing}. "
            "Run 07_SNCounts_DESC_TaskForce_demo.ipynb for this run/configuration first."
        )
    print(f"[{run_name}] loaded cached maps (hard_dust_cut = {metric_ebv_cut(run_name):.3f})")
    return {kind: load_map(p) for kind, p in paths.items()}


results = {run_name: load_run(run_name) for run_name, _ in RUNS_INFO}

nsn_maps = {r: results[r]["nsn"] for r in RUN_NAMES}
zlim_maps = {r: results[r]["zlim"] for r in RUN_NAMES}
KINDS = {
    "NSN": dict(
        maps=nsn_maps,
        label="SNNSNMetric: n_sn (well-measured SNeIa per pixel)",
        unit="SNe / pixel",
        fill_masked=0.0,
    ),
    "ZLIM": dict(
        maps=zlim_maps, label="SNNSNMetric: zlim (redshift completeness limit)", unit="zlim", fill_masked=None
    ),
}

## 7. Per-run summary

`area_deg2` is the area of the pixels where the metric is defined (masked pixels fail the light-curve selection or the `hard_dust_cut`); `sum` for `n_sn` is the **total number of well-measured SNeIa** expected over the footprint - the headline DESC SN count, matching `Total detected` of `03_SN_DESC_TaskForce_demo.ipynb`.

In [ ]:
summaries = {}
summaries["NSN"] = per_run_summary(nsn_maps, with_sum=True)
summaries["ZLIM"] = per_run_summary(zlim_maps, with_sum=False)
for kind, info in KINDS.items():
    summaries[kind].to_csv(join(data_dir, f"SNNSN_{kind}_per_run_summary.csv"))
    print(f'{info["label"]} ({info["unit"]}):')
    display(summaries[kind].round(3))

## 8. Healpix maps for the 7 footprint variants

Grey pixels are masked (rejected by the light-curve selection or `hard_dust_cut`).

In [ ]:
plot_all_maps(
    nsn_maps,
    "SNNSN_NSN",
    "n_sn [SNe / pixel]",
    "SNNSNMetric: n_sn (well-measured SNeIa per pixel)",
    cmap="viridis",
)
plot_all_maps(zlim_maps, "SNNSN_ZLIM", "zlim", "SNNSNMetric: redshift completeness limit zlim", cmap="magma")

## 9. Histograms

In [ ]:
plot_overlay_hist(
    nsn_maps,
    "SNNSN_NSN",
    "n_sn [SNe / pixel]",
    "n_sn: area-weighted histograms (left) and area with value >= x (right)",
)
plot_overlay_hist(
    zlim_maps, "SNNSN_ZLIM", "zlim", "zlim: area-weighted histograms (left) and area with value >= x (right)"
)

## 10. Differences between consecutive dust thresholds

For each pair the map is `value(run_b) - value(run_a)` with `run_b` the run with the *larger* threshold. `n_sn` is extensive (a count): a masked pixel counts as **0**, so the map and the table also capture what is gained or lost with the footprint (`fill_masked=0.0`, `with_totals=True`, as for the WL metrics of notebook 03). `zlim` is intensive (a redshift limit with no natural zero): the difference is restricted to pixels valid in **both** runs (`fill_masked=None`, as for the depth of notebook 01).

In [ ]:
print("#" * 30, "n_sn")
diff_nsn, stats_nsn = analyse_pairs(
    nsn_maps, "SNNSN_NSN", "n_sn", "\u0394 n_sn", "\u0394 n_sn", fill_masked=0.0, with_totals=True
)
display(stats_nsn.round(2))

In [ ]:
print("#" * 30, "zlim")
diff_zlim, stats_zlim = analyse_pairs(
    zlim_maps,
    "SNNSN_ZLIM",
    "zlim",
    "\u0394 zlim",
    "\u0394 zlim (pixels valid in both runs)",
    fill_masked=None,
    with_totals=False,
)
display(stats_zlim.round(4))

## 11. Total `n_sn` and footprint area versus the E(B-V) threshold, with the normalized version

Left: total number of well-measured SNeIa (the headline DESC SN count) and the usable footprint area. Right: same quantities, normalized to the baseline run.

In [ ]:
dust_values = [d for _, d in RUNS_INFO]
i_ref = RUN_NAMES.index(REF_RUN)
totals = np.array([msum(nsn_maps[r]) for r in RUN_NAMES])
areas = np.array([nsn_maps[r].count() * pix_area for r in RUN_NAMES])

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].plot(dust_values, totals, marker="o", color="tab:red")
axs[0].set_ylabel("Total well-measured SNeIa (n_sn)")
axs[1].plot(dust_values, areas, marker="o", color="tab:green")
axs[1].set_ylabel("Footprint area [deg\u00b2]")
for ax in axs:
    ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
    ax.grid(alpha=0.3)
fig.suptitle("SNNSNMetric: total SN count and footprint area versus dust threshold")
fig.tight_layout()
save_fig(fig, "SNNSN_totals_vs_dust_threshold")
plt.show()

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
axs[0].plot(dust_values, totals / totals[i_ref], marker="o", color="tab:red")
axs[0].set_ylabel("Total n_sn / baseline")
axs[1].plot(dust_values, areas / areas[i_ref], marker="o", color="tab:green")
axs[1].set_ylabel("Footprint area / baseline")
for ax in axs:
    ax.axhline(1.0, color="gray", ls="--", linewidth=0.8)
    ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
    ax.grid(alpha=0.3)
fig.suptitle("Same quantities, normalized to the baseline run")
fig.tight_layout()
save_fig(fig, "SNNSN_totals_vs_dust_threshold_norm")
plt.show()

## 12. `zlim` versus the E(B-V) threshold, with the pixel-to-pixel dispersion

Mean (solid) and median (dashed) redshift completeness limit over the footprint of each run, as a function of the E(B-V) threshold. The light band is the mean \u00b1 the standard deviation of the pixel values (dispersion over the sky, not an uncertainty on the mean). The open circle marks the baseline.

In [ ]:
x = np.array([DUST_THRESH[r] for r in RUN_NAMES])
vals = [zlim_maps[r].compressed() for r in RUN_NAMES]
mean = np.array([v.mean() if v.size else np.nan for v in vals])
median = np.array([np.median(v) if v.size else np.nan for v in vals])
std = np.array([v.std() if v.size else np.nan for v in vals])

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.fill_between(
    x, mean - std, mean + std, color="tab:purple", alpha=0.15, linewidth=0, label="mean \u00b1 std"
)
ax.plot(x, mean, "o-", color="tab:purple", label="mean")
ax.plot(x, median, "s--", color="tab:purple", label="median")
ax.plot(x[i_ref], mean[i_ref], "o", mfc="none", mec="k", ms=12, label="baseline")
ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
ax.set_ylabel("zlim")
ax.set_title("SNNSNMetric: redshift completeness limit versus dust threshold")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, "SNNSN_zlim_mean_vs_dust_threshold")
plt.show()

### 12.1 Same plot, normalized to the baseline mean

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(x, mean / mean[i_ref], "o-", color="tab:purple", label="mean")
ax.plot(x, median / median[i_ref], "s--", color="tab:purple", label="median")
ax.axhline(1.0, color="gray", ls="--", linewidth=0.8)
ax.plot(x[i_ref], 1.0, "o", mfc="none", mec="k", ms=12, label="baseline")
ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
ax.set_ylabel("zlim / baseline mean")
ax.set_title("Same quantity, normalized to the baseline mean")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, "SNNSN_zlim_mean_vs_dust_threshold_norm")
plt.show()

## 13. `n_sn` versus `zlim` on the baseline run

Is the SN count driven by the footprint area, the depth (`zlim`), or both? Same diagnostic as Section 8 of `03_SN_DESC_TaskForce_demo.ipynb`, here at `nside=32` instead of the official `nside=16`.

In [ ]:
nsn_vals = nsn_maps[REF_RUN].filled(np.nan)
zlim_vals = zlim_maps[REF_RUN].filled(np.nan)
good = np.isfinite(nsn_vals) & np.isfinite(zlim_vals) & (nsn_vals > 0)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(zlim_vals[good], nsn_vals[good], s=6, alpha=0.3, c="teal")
ax.set_xlabel("Redshift completeness limit, zlim (per pixel)")
ax.set_ylabel("Number of well-measured SNeIa, n_sn (per pixel)")
ax.set_title(f"SNNSNMetric - {REF_RUN} (baseline)")
ax.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, "SNNSN_nsn_vs_zlim_baseline")
plt.show()

## Caveats

- This notebook never reruns `SNNSNMetric`: it trusts the `.npz` caches of `data_07_SNCOUNTS/` to have been produced with exactly the configuration reproduced in Section 3 (`nside`, `n_bef`, `n_aft`, `zmin`/`zmax`, `hard_dust_cut`). If Section 3 is edited here without also rerunning notebook 07 with the matching values, Section 6 will simply fail to find the cache (`FileNotFoundError`) rather than silently mixing configurations.
- **`nside = 32`** was chosen for notebook 07 (finer than the official `nside = 16` of `science_radar_batch`, coarser than the `nside = 64` used elsewhere in this series) as a compromise between spatial resolution and run time across 7 simulations; absolute `n_sn` totals and `zlim` values are expected to differ somewhat from a run at the official `nside = 16` (see `03_SN_DESC_TaskForce_demo.ipynb`) because of the different pixel size, not because of an error in either notebook.
- `n_sn` and `zlim` are computed independently per season inside `SNNSNMetric.run`, then `n_sn` is summed and `zlim` is the median over seasons with `zcomp > 0`; pixels/seasons with too few usable observations, or with `E(B-V)` above `hard_dust_cut`, are masked out entirely rather than contributing a zero (matching `03_SN_DESC_TaskForce_demo.ipynb`).
- The `zmin=0.2, zmax=0.5` window and `n_bef=3` / `n_aft=8` selection are the *official* WFD (Wide-Fast-Deep) entry of `science_radar_batch` - a shallow, low-redshift, WFD-optimized sample, not the deep DDF-based SN cosmology sample (different criteria, much smaller but far more intensively observed footprint; see Gris et al. 2023).
- **SQL wildcard.** Notebook 07 (and hence this cache) uses `scheduler_note not like 'DD%'` (the convention of notebooks 01/03/06 of this series); `03_SN_DESC_TaskForce_demo.ipynb` uses the wider `'%DD%'`. The two should be equivalent for the standard `scheduler_note` values of these simulations, but this has not been independently verified for every run; check the DDF exclusion if the results look inconsistent between the two notebooks.
- With `EBV_CUT_MODE = 'run'`, `hard_dust_cut` follows the threshold of each run, so the metric footprint approximately reproduces the WFD footprint scheduled in that run (same caveat as notebooks 01/03/06 regarding the boundary pixels).
- `0.199` and `baseline (0.200)` have almost the same nominal footprint: the difference between their `n_sn`/`zlim` values gives the scale of the run-to-run (scheduler) fluctuations, to compare against the change seen between the other, more widely separated thresholds.
- The 10-year truncation applies to every run, so the numbers are not those of the full `baseline_v5.3.6_11yrs` simulation.

## References
- Gris, Ph. et al. 2023, "Designing an Optimal LSST Deep Drilling Program for Cosmology with Type Ia Supernovae", ApJS 264, 22 - defines the well-measured-light-curve selection criteria implemented by `SNNSNMetric`.
- Bianco, F. B. et al. 2022, ApJS 258, 1 - SCOC context.
- `rubin_sim.maf` documentation: https://rubin-sim.lsst.io/maf.html
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/metrics/sn_n_sn_metric.py`, `rubin_sim/maf/batches/science_radar_batch.py`)
- `07_SNCounts_DESC_TaskForce_demo.ipynb` - the notebook that ran `SNNSNMetric` and produced the `.npz` cache read here (this notebook's direct source; rerun it if a run is missing or the configuration changes).
- `../06_MAF_DESC_TaskF/03_SN_DESC_TaskForce_demo.ipynb` - the `SNNSNMetric` chain, baseline only, official `nside=16`.
- `01_compareExgalM5withCuts.ipynb`, `03_WeakLensing.ipynb`, `06_FOM3x2pts_DESC_TaskForce_demo.ipynb` - the E(B-V) footprint-threshold comparison for the other DESC metrics of this series.
- `shrink_fp_dust_*.db` simulations: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/shrink_fp/